In [ ]:
import os
os.environ["VCC_NPZ_IN"] = "/kaggle/input/<slug-with-v5g0_vcc_accum_v1.npz>"
os.environ["VCC_OUT"] = "/kaggle/working"

In [ ]:
#!/usr/bin/env python3
"""
v5 G0 — VCC leg. STEP 2: decomposition, effect-size stratification, k, verdict.

Input : v5g0_vcc_accum_v1.npz   (from STEP 1)
Output: v5g0_vcc_reproducibility_v1.csv     component x depth x seed x bin
        v5g0_vcc_perpert_v1.csv             per-perturbation r, DEG count, bin
        v5g0_vcc_variance_decomp_v1.csv
        v5g0_vcc_deg_per_pert_v1.csv
        v5g0_vcc_kfit_v1.csv                k overall and per bin, with CIs
        v5g0_vcc_psweep_v1.csv              P in {6,25,150} sanity check
        v5g0_vcc_report_v1.json             headline + LOCKED VERDICT
        v5g0_vcc_depth_curve_v1.png         measured points + fitted 1/(1+k/n)

DECOMPOSITION (Brian, locked):
    d_p = x_p - x_ntc ;  g = mean_p d_p  (RECOMPUTED INSIDE EACH HALF) ;  s_p = d_p - g

PRIMARY DELIVERABLE: k, where  r_specific(n) = 1 / (1 + k/n).
    k = cells per perturbation at which the specific component reaches r = 0.5.
    scBloodNL reference: k ~= 457.

PRE-REGISTERED DECISION RULE (§9 of docs/TASK_v5_G0_VCC_leg.md, locked):
    INVALID        global < 0.8                  -> pipeline bug, report nothing else
    PARADIGM       k > 2848                      (r < 0.30 at n = 1221)
    AMBIGUOUS      814 <= k <= 2848              (r ~ 0.45 at k ~ 1492)
    MODELING FORK  k < 814                       (r > 0.60 at n = 1221)

DEPTH CONVENTION — READ THIS: as in the scBloodNL leg, n = cells per HALF, so a
depth of n consumes 2n cells. VCC's ~1,221 cells/perturbation therefore affords
~610 per half. r at n = 1221 is an EXTRAPOLATION from the fitted k and is
reported as such; k itself is measured, convention-independent, and is what the
§9 thresholds are stated in.
"""
import json
import os

import numpy as np
import pandas as pd

OUT = os.environ.get("VCC_OUT", "/kaggle/working")
# NPZ_IN lets the input (e.g. a read-only /kaggle/input/<dataset> attached from a
# prior step-1 run) live somewhere different from OUT (where this script writes
# results). Defaults to OUT so existing single-folder usage is unaffected.
NPZ_IN = os.environ.get("VCC_NPZ_IN", OUT)
NPZ = os.path.join(NPZ_IN, "v5g0_vcc_accum_v1.npz")
if not os.path.exists(NPZ):
    # fall back to an auto-discovery search under /kaggle/input in case the
    # dataset slug puts the file in a nested folder
    import glob
    hits = glob.glob("/kaggle/input/**/v5g0_vcc_accum_v1.npz", recursive=True)
    if hits:
        NPZ = hits[0]
    else:
        raise FileNotFoundError(
            f"v5g0_vcc_accum_v1.npz not found at {NPZ} or under /kaggle/input. "
            "Set VCC_NPZ_IN to the folder containing it (e.g. the attached "
            "step-1 output dataset), or VCC_OUT if it's already in /kaggle/working.")
os.makedirs(OUT, exist_ok=True)
print(f"reading npz from: {NPZ}")
print(f"writing outputs to: {OUT}")

MIN_DETECT_FRAC = 0.01
FDR = 0.05
DEG_STRONG, DEG_SUBTLE = 100, 10       # Arc's own bins: >100 / 10-100 / <10
VCC_NATIVE_N = 1221                    # cells per perturbation quoted in the brief
K_PARADIGM, K_FORK = 2848.0, 814.0
N_BOOT = int(os.environ.get("VCC_NBOOT", "500"))

z = np.load(NPZ, allow_pickle=True)
accum = z["accum"].astype(np.float64)          # (run, bin, gene)
bin_n = z["bin_n"]                             # (run, bin)
detect = z["detect"]
genes = z["genes"]
groups = list(z["groups"])
DEPTHS = list(z["depths"])
N_SLOT, N_HALF, N_GROUP = int(z["n_slot"]), int(z["n_half"]), int(z["n_group"])
run_kind = list(z["run_kind"])
run_seed = list(z["run_seed"])
avail = z["avail"]                             # (run, group, half)
N_CELLS = int(z["n_cells"])
X_MODE = str(z["x_mode"])
CTRL_LABEL = str(z["ctrl_label"])
deg_n, deg_sum, deg_sumsq = z["deg_n"], z["deg_sum"], z["deg_sumsq"]

CTRL, PERT_IDX = 0, list(range(1, N_GROUP))
N_GENE = len(genes)
gene_mask = (detect / N_CELLS) >= MIN_DETECT_FRAC
NG = int(gene_mask.sum())
print(f"{NG}/{N_GENE} genes pass detect >= {MIN_DETECT_FRAC:.0%}")


def bin_id(g, half, slot):
    return (g * N_HALF + half) * N_SLOT + slot


def corr(a, b):
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


# ------------------------------------------- effect size: DEGs per pert -----
def bh(p):
    p = np.asarray(p, dtype=float)
    ok = np.isfinite(p)
    q = np.full(p.size, np.nan)
    pv = p[ok]
    o = np.argsort(pv)
    n = pv.size
    adj = np.minimum.accumulate((pv[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    tmp = np.empty(n); tmp[o] = np.clip(adj, 0, 1)
    q[ok] = tmp
    return q


from scipy import stats

mean_g = deg_sum / np.maximum(deg_n, 1)[:, None]
var_g = deg_sumsq / np.maximum(deg_n, 1)[:, None] - mean_g ** 2
var_g = np.maximum(var_g, 0) * (deg_n / np.maximum(deg_n - 1, 1))[:, None]

def welch_ndeg(m1, v1, n1, m0, v0, n0):
    se2 = v1 / max(n1, 1) + v0 / max(n0, 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        t = (m1 - m0) / np.sqrt(se2)
        df = se2 ** 2 / ((v1 / max(n1, 1)) ** 2 / max(n1 - 1, 1) +
                         (v0 / max(n0, 1)) ** 2 / max(n0 - 1, 1))
    p = 2 * stats.t.sf(np.abs(t), np.clip(df, 1, None))
    return int(np.sum(bh(np.where(np.isfinite(p), p, 1.0)) < FDR))


def label_bin(nd):
    return "strong" if nd > DEG_STRONG else "subtle" if nd >= DEG_SUBTLE else "negligible"


def profile(ri, g, part, slot):
    """Cumulative pseudobulk through depth slot `slot`, on masked genes."""
    s = accum[ri, bin_id(g, part, 0):bin_id(g, part, slot) + 1].sum(0)[gene_mask]
    n = bin_n[ri, bin_id(g, part, 0):bin_id(g, part, slot) + 1].sum()
    if n == 0:
        return np.full(NG, np.nan)
    if X_MODE == "RAW_COUNTS":
        tot = s.sum()
        return np.log1p(s / tot * 1e6) if tot > 0 else np.full(NG, np.nan)
    return s / n                     # already log-space -> mean log expression


# TWO binning schemes, both computed on the HELD-OUT PANEL C — cells that appear
# in neither A nor B. The stratification is therefore independent of the split-half
# reproducibility it stratifies; no selection bias, no circularity.
#   vs_ntc   — Arc's own convention (DEGs vs non-targeting), comparable to their
#              35/29/46 split. PRIMARY. It counts global + specific effect, so a
#              strong shared program can push every perturbation into "strong";
#              if that happens the scheme has collapsed and spec_mag carries it.
#   spec_mag — terciles of ||s_p^C||, the magnitude of the SPECIFIC component on
#              panel C. Directly stratifies the quantity being measured.
deg_rows = []
m0, v0, n0 = mean_g[CTRL][gene_mask], var_g[CTRL][gene_mask], deg_n[CTRL]
dC = {g: profile(0, g, 2, 0) - profile(0, CTRL, 2, 0) for g in PERT_IDX}
gC = np.mean([dC[g] for g in PERT_IDX], axis=0)
spec_mag = {groups[g]: float(np.sqrt(np.nansum((dC[g] - gC) ** 2))) for g in PERT_IDX}
mag_vals = np.array([spec_mag[groups[g]] for g in PERT_IDX])
t1, t2 = np.nanpercentile(mag_vals, [33.333, 66.667])
for g in PERT_IDX:
    m1, v1, n1 = mean_g[g][gene_mask], var_g[g][gene_mask], deg_n[g]
    nd_ntc = welch_ndeg(m1, v1, n1, m0, v0, n0)
    mg = spec_mag[groups[g]]
    deg_rows.append(dict(perturbation=groups[g], n_cells_panelC=int(deg_n[g]),
                         n_deg_vs_ntc=nd_ntc, bin_vs_ntc=label_bin(nd_ntc),
                         specific_magnitude_panelC=mg,
                         bin_spec_mag=("strong" if mg > t2 else
                                       "subtle" if mg > t1 else "negligible")))
deg_df = pd.DataFrame(deg_rows)
SCHEMES = {"vs_ntc": dict(zip(deg_df.perturbation, deg_df.bin_vs_ntc)),
           "spec_mag": dict(zip(deg_df.perturbation, deg_df.bin_spec_mag))}
bin_of = SCHEMES["vs_ntc"]
deg_of = dict(zip(deg_df.perturbation, deg_df.n_deg_vs_ntc))
mag_of = dict(zip(deg_df.perturbation, deg_df.specific_magnitude_panelC))
print("vs_ntc  :", deg_df.bin_vs_ntc.value_counts().to_dict())
print("spec_mag:", deg_df.bin_spec_mag.value_counts().to_dict(),
      f"(tercile cuts {t1:.3f} / {t2:.3f})")


# ------------------------------------------------------------- profiles -----
AB = (0, 1)                          # partition 2 is the held-out effect-size panel C
rows, perpert_rows, var_rows = [], [], []
# balanced panel: perturbations that survive to the deepest grid point in every run
deepest = DEPTHS[-1]
balanced = [g for g in PERT_IDX
            if all(min(avail[ri][g][0], avail[ri][g][1]) >= deepest for ri in range(len(run_kind)))]
print(f"balanced panel: {len(balanced)}/{len(PERT_IDX)} perturbations reach {deepest} cells/half")

for ri, (kind, seed) in enumerate(zip(run_kind, run_seed)):
    for slot, depth in enumerate(DEPTHS):
        incl = [g for g in balanced if min(avail[ri][g][0], avail[ri][g][1]) >= depth]
        if len(incl) < 5 or min(avail[ri][CTRL][0], avail[ri][CTRL][1]) < depth:
            continue
        pc = {h: profile(ri, CTRL, h, slot) for h in AB}
        d = {(g, h): profile(ri, g, h, slot) - pc[h] for g in incl for h in AB}
        gA = np.mean([d[(g, 0)] for g in incl], axis=0)
        gB = np.mean([d[(g, 1)] for g in incl], axis=0)
        sA = {g: d[(g, 0)] - gA for g in incl}
        sB = {g: d[(g, 1)] - gB for g in incl}

        per = []
        for g in incl:
            rt, rs = corr(d[(g, 0)], d[(g, 1)]), corr(sA[g], sB[g])
            lab = groups[g]
            per.append((lab, rt, rs))
            perpert_rows.append(dict(dataset="VCC_adata_Training", perturbation=lab,
                                     run_kind=kind, seed=int(seed), cells_per_pert=depth,
                                     r_total=rt, r_specific=rs,
                                     n_deg_vs_ntc=deg_of.get(lab),
                                     specific_magnitude=mag_of.get(lab),
                                     bin_vs_ntc=SCHEMES["vs_ntc"].get(lab),
                                     bin_spec_mag=SCHEMES["spec_mag"].get(lab)))
        r_glob = corr(gA, gB)

        def emit(subset, label, scheme):
            if not subset:
                return
            for comp, j in (("total", 1), ("specific", 2)):
                rows.append(dict(dataset="VCC_adata_Training", run_kind=kind, seed=int(seed),
                                 cells_per_pert=depth, bin_scheme=scheme, effect_bin=label,
                                 component=comp,
                                 reproducibility=float(np.nanmean([x[j] for x in subset])),
                                 n_perturbations=len(subset), n_genes=NG))

        for scheme, mapping in SCHEMES.items():
            emit(per, "all", scheme)
            for b in ("strong", "subtle", "negligible"):
                emit([x for x in per if mapping.get(x[0]) == b], b, scheme)
            rows.append(dict(dataset="VCC_adata_Training", run_kind=kind, seed=int(seed),
                             cells_per_pert=depth, bin_scheme=scheme, effect_bin="all",
                             component="global", reproducibility=r_glob,
                             n_perturbations=len(incl), n_genes=NG))

        for half, (gv, sv) in enumerate([(gA, sA), (gB, sB)]):
            dv = [d[(g, half)] for g in incl]
            tot_v = float(np.mean([np.sum(x ** 2) for x in dv]))
            glob_v = float(np.sum(gv ** 2))
            spec_v = float(np.mean([np.sum(sv[g] ** 2) for g in incl]))
            var_rows.append(dict(run_kind=kind, seed=int(seed), cells_per_pert=depth, half=half,
                                 var_total=tot_v, var_global=glob_v, var_specific=spec_v,
                                 frac_global=glob_v / tot_v if tot_v else np.nan,
                                 frac_specific=spec_v / tot_v if tot_v else np.nan))
    print(f"  run {kind} seed {seed} done")

rep = pd.DataFrame(rows)
pp = pd.DataFrame(perpert_rows)
vd = pd.DataFrame(var_rows)
rep.to_csv(os.path.join(OUT, "v5g0_vcc_reproducibility_v1.csv"), index=False)
pp.to_csv(os.path.join(OUT, "v5g0_vcc_perpert_v1.csv"), index=False)
vd.to_csv(os.path.join(OUT, "v5g0_vcc_variance_decomp_v1.csv"), index=False)
deg_df.to_csv(os.path.join(OUT, "v5g0_vcc_deg_per_pert_v1.csv"), index=False)


# ------------------------------------------------------------------ k fit ---
def fit_k(n, r):
    """Least squares on r(n) = 1/(1+k/n). Grid + golden refine; no scipy dep,
    no fitting freedom beyond the single parameter."""
    n = np.asarray(n, float); r = np.asarray(r, float)
    ok = np.isfinite(n) & np.isfinite(r)
    n, r = n[ok], r[ok]
    if n.size < 2:
        return np.nan
    grid = np.exp(np.linspace(np.log(1.0), np.log(1e6), 4000))
    sse = ((1.0 / (1.0 + grid[:, None] / n[None, :]) - r[None, :]) ** 2).sum(1)
    return float(grid[int(np.argmin(sse))])


def k_from_point(n, r):
    return n * (1 - r) / r if (np.isfinite(r) and r > 0) else np.nan


krows = []
real = rep[rep.run_kind == "real"]
for scheme in SCHEMES:
    col = "bin_vs_ntc" if scheme == "vs_ntc" else "bin_spec_mag"
    for label in ("all", "strong", "subtle", "negligible"):
        sub = real[(real.bin_scheme == scheme) & (real.effect_bin == label) &
                   (real.component == "specific")]
        if sub.empty:
            continue
        curve = sub.groupby("cells_per_pert").reproducibility.mean()
        k_hat = fit_k(curve.index.values, curve.values)

        # bootstrap over perturbations (the unit of replication)
        sel = pp[pp.run_kind == "real"]
        if label != "all":
            sel = sel[sel[col] == label]
        piv = sel.pivot_table(index="perturbation", columns="cells_per_pert", values="r_specific")
        rng = np.random.default_rng(0)
        boots = []
        for _ in range(N_BOOT):
            m = piv.iloc[rng.integers(0, piv.shape[0], piv.shape[0])].mean(0)
            kb = fit_k(m.index.values.astype(float), m.values)
            if np.isfinite(kb):
                boots.append(kb)
        lo, hi = ((float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)))
                  if boots else (np.nan, np.nan))

        krows.append(dict(bin_scheme=scheme, effect_bin=label,
                          n_perturbations=int(piv.shape[0]), k=k_hat, k_ci_lo=lo, k_ci_hi=hi,
                          k_over_scbloodnl=k_hat / 457.0,
                          r_specific_at_n1221_extrapolated=1.0 / (1.0 + k_hat / VCC_NATIVE_N),
                          r_specific_at_n610_achievable=1.0 / (1.0 + k_hat / (VCC_NATIVE_N / 2)),
                          per_depth_k={int(n_): round(k_from_point(n_, r_), 1)
                                       for n_, r_ in curve.items()},
                          measured=dict(zip(curve.index.astype(int).tolist(),
                                            np.round(curve.values, 4).tolist()))))
kfit = pd.DataFrame(krows)
kfit.to_csv(os.path.join(OUT, "v5g0_vcc_kfit_v1.csv"), index=False)
print(kfit[["bin_scheme", "effect_bin", "n_perturbations", "k", "k_ci_lo", "k_ci_hi",
            "r_specific_at_n1221_extrapolated"]].to_string(index=False))


# ------------------------------------------------ P-sweep sanity check ------
psweep = []
slot = len(DEPTHS) - 1
depth = DEPTHS[slot]
for ri, (kind, seed) in enumerate(zip(run_kind, run_seed)):
    if kind != "real":
        continue
    incl_all = [g for g in balanced if min(avail[ri][g][0], avail[ri][g][1]) >= depth]
    if len(incl_all) < 6:
        continue
    pc = {h: profile(ri, CTRL, h, slot) for h in AB}
    dcache = {(g, h): profile(ri, g, h, slot) - pc[h] for g in incl_all for h in AB}
    rng = np.random.default_rng(seed)
    for P in (6, 25, 150):
        if P > len(incl_all):
            P = len(incl_all)
        vals = []
        for _ in range(20):
            sub = list(rng.choice(incl_all, size=P, replace=False))
            gA = np.mean([dcache[(g, 0)] for g in sub], axis=0)
            gB = np.mean([dcache[(g, 1)] for g in sub], axis=0)
            vals += [corr(dcache[(g, 0)] - gA, dcache[(g, 1)] - gB) for g in sub]
        psweep.append(dict(seed=int(seed), P=P, cells_per_pert=depth,
                           r_specific=float(np.nanmean(vals))))
ps = pd.DataFrame(psweep)
ps.to_csv(os.path.join(OUT, "v5g0_vcc_psweep_v1.csv"), index=False)


# ------------------------------------------------------- verdict (locked) ---
glob = float(real[(real.component == "global") & (real.bin_scheme == "vs_ntc")]
             .reproducibility.mean())
null = rep[(rep.run_kind == "shuffled") & (rep.bin_scheme == "vs_ntc")]
null_spec = float(null[(null.component == "specific") & (null.effect_bin == "all")].reproducibility.mean())
null_glob = float(null[null.component == "global"].reproducibility.mean())
kall_row = kfit[(kfit.bin_scheme == "vs_ntc") & (kfit.effect_bin == "all")].iloc[0]
k_all, k_lo, k_hi = float(kall_row.k), float(kall_row.k_ci_lo), float(kall_row.k_ci_hi)

if glob < 0.8:
    verdict, why = "INVALID", f"global reproducibility {glob:.4f} < 0.80 -> pipeline bug, not biology."
elif k_all > K_PARADIGM:
    verdict, why = "PARADIGM", f"k = {k_all:.0f} > {K_PARADIGM:.0f} (r={1/(1+k_all/VCC_NATIVE_N):.3f} at n={VCC_NATIVE_N})."
elif k_all < K_FORK:
    verdict, why = "MODELING FORK", f"k = {k_all:.0f} < {K_FORK:.0f} (r={1/(1+k_all/VCC_NATIVE_N):.3f} at n={VCC_NATIVE_N})."
else:
    verdict, why = "AMBIGUOUS", f"k = {k_all:.0f} sits in [{K_FORK:.0f}, {K_PARADIGM:.0f}]."

bounce = []
if glob < 0.8:
    bounce.append(f"global {glob:.4f} < 0.8 -> pipeline failure, not a finding")
if abs(null_spec) > 0.1:
    bounce.append(f"shuffled null specific = {null_spec:.4f}, not ~0 -> decomposition leaking")
if np.isfinite(k_lo) and (k_lo < K_FORK < k_hi or k_lo < K_PARADIGM < k_hi):
    bounce.append(f"k CI [{k_lo:.0f}, {k_hi:.0f}] straddles a §9 threshold -> do not force the verdict")
seed_k = real[(real.bin_scheme == "vs_ntc") & (real.effect_bin == "all") &
              (real.component == "specific")]
sk = [fit_k(gr.cells_per_pert.values, gr.reproducibility.values)
      for _, gr in seed_k.groupby("seed")]
if len(sk) > 1 and (max(sk) - min(sk)) / max(np.mean(sk), 1e-9) > 0.25:
    bounce.append(f"k unstable across seeds: {[round(x) for x in sk]}")
cv = float(np.std(z["counts_per_group"][1:]) / np.mean(z["counts_per_group"][1:]))
if cv > 0.5:
    bounce.append(f"cells-per-perturbation highly non-uniform (CV={cv:.2f}) -> a single n misrepresents the dataset")

report = {
    "task": "v5 G0 — VCC leg (two-component reproducibility, k)",
    "dataset": "adata_Training.h5ad — the exact data the 337-team VCC leaderboard was scored on",
    "x_mode": X_MODE, "control_label": CTRL_LABEL,
    "design": {
        "n_perturbations": len(PERT_IDX), "n_genes_after_filter": NG,
        "gene_filter": f"detected in >= {MIN_DETECT_FRAC:.0%} of cells",
        "normalisation": "sum counts -> CPM -> log1p" if X_MODE == "RAW_COUNTS" else "mean of per-cell log values",
        "depth_convention": "n = cells per HALF; depth n consumes 2n cells",
        "depth_grid": [int(x) for x in DEPTHS],
        "balanced_panel_size": len(balanced),
        "control_split": "non-targeting split into its own A/B halves at matched depth",
        "g_recomputed_within_half": True,
        "real_seeds": [int(s) for k_, s in zip(run_kind, run_seed) if k_ == "real"],
        "null_seeds": [int(s) for k_, s in zip(run_kind, run_seed) if k_ == "shuffled"],
        "cells_per_pert_cv": cv,
    },
    "effect_size_bins": {
        "vs_ntc_arc_comparable": deg_df.bin_vs_ntc.value_counts().to_dict(),
        "spec_mag_terciles_panelC": deg_df.bin_spec_mag.value_counts().to_dict(),
        "arc_reported_split": {"strong": 0.35, "subtle": 0.29, "negligible": 0.46},
        "computed_on": "held-out panel C (disjoint from A and B) -> no selection bias",
    },
    "controls": {
        "global_reproducibility_positive_control": glob,
        "shuffled_null_specific": null_spec,
        "shuffled_null_global": null_glob,
    },
    "k": kfit.drop(columns=["measured"]).to_dict("records"),
    "k_all_ci": [k_lo, k_hi],
    "scbloodnl_reference_k": 457.0,
    "variance_decomposition_real": {
        "frac_global_mean": float(vd[vd.run_kind == "real"].frac_global.mean()),
        "frac_specific_mean": float(vd[vd.run_kind == "real"].frac_specific.mean()),
    },
    "p_sweep_sanity": ps.groupby("P").r_specific.mean().to_dict() if not ps.empty else {},
    "VERDICT": verdict,
    "rationale": why,
    "caveat_depth_convention": (
        f"VCC affords ~{VCC_NATIVE_N//2} cells per half, not {VCC_NATIVE_N}. r at "
        f"n={VCC_NATIVE_N} is extrapolated from the fitted k; k itself is measured."),
    "bounce_back_to_brian": bounce,
}
with open(os.path.join(OUT, "v5g0_vcc_report_v1.json"), "w") as f:
    json.dump(report, f, indent=2, default=str)
print(json.dumps({kk: report[kk] for kk in ("controls", "k_all_ci", "VERDICT", "rationale",
                                            "bounce_back_to_brian")}, indent=2, default=str))

# ---------------------------------------------------------------- figure ----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
ax = axes[0]
colors = {"total": "#333333", "global": "#1f77b4", "specific": "#d62728"}
for comp in ("total", "global", "specific"):
    for kind, ls, mk in (("real", "-", "o"), ("shuffled", "--", "x")):
        sub = (rep[(rep.bin_scheme == "vs_ntc") & (rep.effect_bin == "all") &
                   (rep.component == comp) & (rep.run_kind == kind)]
               .groupby("cells_per_pert").reproducibility.agg(["mean", "std"]).reset_index())
        if sub.empty:
            continue
        ax.errorbar(sub.cells_per_pert, sub["mean"], yerr=sub["std"], ls=ls, marker=mk,
                    color=colors[comp], ms=4, lw=1.5 if kind == "real" else 1,
                    label=f"{comp} ({kind})")
nn = np.exp(np.linspace(np.log(10), np.log(4000), 200))
ax.plot(nn, 1 / (1 + k_all / nn), color="#d62728", lw=1, alpha=.5, ls=":",
        label=f"fit k={k_all:.0f}")
ax.axvline(VCC_NATIVE_N / 2, color="grey", lw=.8)
ax.axvline(VCC_NATIVE_N, color="grey", lw=.8, ls="--")
for h in (0.3, 0.5, 0.6, 0.8):
    ax.axhline(h, color="grey", lw=.5, ls=":")
ax.set_xscale("log"); ax.set_ylim(-0.2, 1.02)
ax.set_xlabel("cells per perturbation (per half)")
ax.set_ylabel("split-half reproducibility (Pearson r)")
ax.set_title("VCC — two-component reproducibility"); ax.legend(fontsize=7, loc="upper left")

for ax, scheme, ttl in ((axes[1], "vs_ntc", "DEGs vs non-targeting (Arc convention)"),
                        (axes[2], "spec_mag", "||s_p|| terciles, held-out panel C)")):
    for b, c in (("strong", "#2ca02c"), ("subtle", "#ff7f0e"),
                 ("negligible", "#9467bd"), ("all", "#d62728")):
        sub = (rep[(rep.bin_scheme == scheme) & (rep.effect_bin == b) &
                   (rep.component == "specific") & (rep.run_kind == "real")]
               .groupby("cells_per_pert").reproducibility.mean())
        if sub.empty:
            continue
        kb = kfit[(kfit.bin_scheme == scheme) & (kfit.effect_bin == b)]
        lbl = f"{b} (k={float(kb.k.iloc[0]):.0f})" if len(kb) else b
        ax.plot(sub.index, sub.values, "o-", color=c, ms=4, label=lbl)
        if len(kb):
            ax.plot(nn, 1 / (1 + float(kb.k.iloc[0]) / nn), color=c, lw=1, ls=":", alpha=.6)
    ax.axvline(VCC_NATIVE_N / 2, color="grey", lw=.8)
    ax.axhline(0.5, color="grey", lw=.5, ls=":")
    ax.set_xscale("log"); ax.set_ylim(-0.1, 1.0)
    ax.set_xlabel("cells per perturbation (per half)")
    ax.set_title(f"specific component — {ttl}", fontsize=9); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "v5g0_vcc_depth_curve_v1.png"), dpi=300, bbox_inches="tight")
print("figure written")